# Import SP Tren Ligero from OSMNX to Visum
(these were retrieved by a query from OSMNX and fall directly on a railway node)

In [22]:
import pandas as pd
import geopandas as gpd
import os

In [50]:
folder = r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara'
sp_osmnx = os.path.join(folder, "Pruebas OSM-OSMNX", "nodes-edges osmnx", "stop points", "stop points metro", "stop_points_metro.shp")

In [51]:
# Read Stop Points Tren Ligero
tren_ligero_SP = gpd.read_file(sp_osmnx)
tren_ligero_SP = tren_ligero_SP[tren_ligero_SP['network'] != "Mi Transporte"] # remove 1 bus stop
tren_ligero_SP['node_id'] = tren_ligero_SP['node_id'].astype(int)
tren_ligero_SP['TRANSPORT'] = "TREN LIGERO"

print(f"Total SP Tren Ligero: {len(tren_ligero_SP)}")

Total SP Tren Ligero: 113


## Open Visum for object creation

In [54]:
import win32com.client as com

latest_network = os.path.join(folder, "Visum Projects", "Jeannette", "RedOSMNX_AMG_24 - Jul.ver")

Visum = com.Dispatch("Visum.Visum")
Visum.LoadVersion(latest_network)
C = com.constants
Net = Visum.Net

In [55]:
stop_nos = Visum.Net.Stops.GetMultiAttValues("No")
stopareas_nos = Visum.Net.StopAreas.GetMultiAttValues("No")
stoppoint_nos = Visum.Net.StopPoints.GetMultiAttValues("No")

last_stop = max(value for _, value in stop_nos)
last_stoparea = max(value for _, value in stopareas_nos)
last_stoppoint = max(value for _, value in stoppoint_nos)

print("Último número de stop:", last_stop)
print("Último número de stop area:", last_stoparea)
print("Último número de stop point:", last_stoppoint)

Último número de stop: 12518.0
Último número de stop area: 12518.0
Último número de stop point: 16381.0


In [56]:
tren_ligero_SP

,element,id,name,network,operator,public_tra,railway,subway,power,tram,...,roof_shape,roof_level,undergroun,baby_feedi,area,aerialway,constructi,TRANSPORT,node_id,geometry
0,node,340135806,La Aurora,Mi Tren,SITEUR,stop_position,stop,yes,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205221,POINT (-103.28557 20.66244)
1,node,340135807,San Jacinto,Mi Tren,SITEUR,stop_position,stop,yes,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205222,POINT (-103.29729 20.66385)
2,node,340135808,San Andrés,Mi Tren,SITEUR,stop_position,stop,yes,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205223,POINT (-103.3061 20.66526)
3,node,340135813,Cristobal de Oñate,Mi Tren,SITEUR,stop_position,stop,yes,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205224,POINT (-103.31339 20.66749)
4,node,340135815,Oblatos,Mi Tren,SITEUR,stop_position,stop,yes,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205225,POINT (-103.32252 20.67026)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,node,12699043507,Jalisco 200 Años,None,SITEUR,stop_position,stop,None,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205996,POINT (-103.35597 20.57503)
110,node,12699043508,Real del Valle,None,SITEUR,stop_position,stop,None,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205997,POINT (-103.36548 20.55948)
111,node,12699043509,Real del Valle,None,SITEUR,stop_position,stop,None,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,205998,POINT (-103.36555 20.55951)
112,node,12952466831,Acueducto,None,SITEUR,stop_position,stop,None,None,None,...,None,None,None,None,None,None,None,TREN LIGERO,206021,POINT (-103.34655 20.5883)


In [57]:
stop_id = last_stoppoint + 1

for row in tren_ligero_SP.itertuples():
    node_id = row.node_id
    transport = row.network if pd.notna(row.network) else row.TRANSPORT

    try:
        # 1) Add stop
        stop = Net.AddStop(stop_id)
        stop.SetAttValue("Name", row.name)
        stop.SetAttValue("TRANSPORT", transport)

        # 2) Add stop area
        stop_area = Net.AddStopArea(stop_id, stop, node_id)
        stop_area.SetAttValue("Name", row.name)
        stop_area.SetAttValue("TRANSPORT", transport)

        # 3) Add stop point
        stop_point = Net.AddStopPointOnNode(stop_id, stop_area, node_id)
        stop_point.SetAttValue("TSysSet", "TL")
        stop_point.SetAttValue("Name", row.name)
        stop_point.SetAttValue("TRANSPORT",  transport)
        
        print(f'stop point added for node_id: {row.node_id}')

    except Exception as e:
        print(f"Error on node {node_id}: {e}")

    stop_id += 1


stop point added for node_id: 205221
stop point added for node_id: 205222
stop point added for node_id: 205223
stop point added for node_id: 205224
stop point added for node_id: 205225
stop point added for node_id: 205226
stop point added for node_id: 205227
stop point added for node_id: 205287
stop point added for node_id: 205431
stop point added for node_id: 205444
stop point added for node_id: 205450
stop point added for node_id: 205452
stop point added for node_id: 205453
stop point added for node_id: 205458
stop point added for node_id: 205461
stop point added for node_id: 205469
stop point added for node_id: 205626
stop point added for node_id: 205627
stop point added for node_id: 205628
stop point added for node_id: 205629
stop point added for node_id: 206308
stop point added for node_id: 206309
stop point added for node_id: 206310
stop point added for node_id: 206311
stop point added for node_id: 206325
stop point added for node_id: 206326
stop point added for node_id: 206327
s